# `PCPFLAREINV` approximate inverses

This notebook focuses on the approximate inverses found in `PCPFLAREINV`. In particular we only focus on the GMRES polynomials, as they are required in future notebooks when we discuss AIRG.

We can write GMRES polynomials in various forms, but first we consider the monomial form
$$
p_m(A)=\sum_{k=0}^{m} c_k A^k,
$$
where coefficients are chosen to minimise the 2-norm residual for a given right-hand side.

A different GMRES polynomial is formed during GMRES at each iteration, which is non-stationary. This requires a dot product at each iteration during the orthogonalisation. `PCPFLAREINV` however builds a fixed order stationary polynomial from a single random right-hand side and reuses it as an approximate inverse.

Key benefit: once coefficients are built, applying the polynomial uses only matrix-vector products.

`PCPFLAREINV` defaults used here:
- Type: GMRES polynomial (`arnoldi`)
- Polynomial order: 6
- Assembled application with fixed sparsity order 1

In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc

import pflare

print('Ready.')

## 1. Build a test matrix

We use a simple structured 2D upwind advection operator as the test problem.

In [ ]:
def build_2d_upwind(N):
    """Pure 2D upwind advection u_x + u_y = 1 on [0,1]^2."""
    h = 1.0 / (N + 1)
    n = N * N

    A = PETSc.Mat().create()
    A.setSizes([n, n])
    A.setFromOptions()
    A.setPreallocationNNZ(3)
    A.setUp()

    rstart, rend = A.getOwnershipRange()
    for row in range(rstart, rend):
        i, j = divmod(row, N)
        A.setValue(row, row, 2.0 / h)
        if j > 0:
            A.setValue(row, row - 1, -1.0 / h)
        if i > 0:
            A.setValue(row, row - N, -1.0 / h)

    A.assemblyBegin()
    A.assemblyEnd()

    b = A.createVecRight()
    b.set(1.0)
    return A, b

N = 40
A, b = build_2d_upwind(N)
n = A.getSize()[0]
nnz_A = A.getInfo()['nz_used']
print(f'A: size={n}, nnz={nnz_A:.0f}')

## 2. Utilities

When doing comparisons vs GMRES polynomials, either assembled or matrix-free, we must be careful to compare the equivalent matrix-vector costs. For assembled matrices one application of the approximate inverse costs as many nnzs as we keep given the fixed sparsity. For matrix-free matrices, one application of the approximate inverse costs one matvec of the original matrix for each order.

For consistency across all examples, all KSP solves in this notebook use **unpreconditioned residual norms** for both monitoring and stopping tests.

In [ ]:
def _set_prefixed_options(prefix, options_dict):
    opts = PETSc.Options()
    for key, value in options_dict.items():
        full_key = prefix + key
        opts[full_key] = str(value)

def _clear_prefixed_options(prefix, options_dict):
    opts = PETSc.Options()
    for key in options_dict.keys():
        full_key = prefix + key
        try:
            del opts[full_key]
        except Exception:
            pass

def run_solver_with_history(
    A, b, ksp_type='gmres', pc_type='none', pflare_opts=None,
    max_it=200, rtol=1e-10
):
    """Run KSP and return relative residual history and iteration count."""
    pflare_opts = pflare_opts or {}
    prefix = f'smooth_{int(time.time()*1e6)%10_000_000}_'

    ksp = PETSc.KSP().create()
    ksp.setOptionsPrefix(prefix)
    ksp.setOperators(A)
    ksp.setType(ksp_type)
    ksp.setTolerances(rtol=rtol, max_it=max_it)
    ksp.getPC().setType(pc_type)

    _set_prefixed_options(prefix, pflare_opts)

    rnorms = []
    def monitor(ksp_obj, it, rnorm):
        rnorms.append(rnorm)
    ksp.setMonitor(monitor)

    ksp.setFromOptions()
    ksp.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)
    x = A.createVecRight()
    ksp.solve(b, x)

    _clear_prefixed_options(prefix, pflare_opts)

    rnorms = np.array(rnorms, dtype=float)
    if rnorms.size > 0 and rnorms[0] != 0.0:
        rrel = rnorms / rnorms[0]
    else:
        rrel = rnorms
    return rrel, ksp.getIterationNumber(), ksp

class FixedGMRESApplyPC:
    """Python PC that applies a fixed number of GMRES iterations as M^{-1}."""
    def __init__(self, A, gmres_steps=6):
        self.A = A
        self.gmres_steps = int(gmres_steps)
        self.inner = PETSc.KSP().create(comm=A.getComm())
        self.inner.setOperators(A)
        self.inner.setType('gmres')
        self.inner.getPC().setType('none')
        self.inner.setGMRESRestart(self.gmres_steps)
        self.inner.setTolerances(rtol=0.0, atol=0.0, max_it=self.gmres_steps)
        self.inner.setFromOptions()
        self.inner.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)

    def apply(self, pc, x, y):
        y.set(0.0)
        self.inner.solve(x, y)

def run_richardson_with_fixed_gmres_apply(A, b, gmres_steps=6, max_it=200, rtol=1e-10):
    """Outer undamped Richardson with inner fixed-step GMRES apply as preconditioner."""
    ksp = PETSc.KSP().create(comm=A.getComm())
    ksp.setOperators(A)
    ksp.setType('richardson')
    ksp.setTolerances(rtol=rtol, max_it=max_it)

    pc = ksp.getPC()
    pc.setType(PETSc.PC.Type.PYTHON)
    pc.setPythonContext(FixedGMRESApplyPC(A, gmres_steps=gmres_steps))

    rnorms = []
    def monitor(ksp_obj, it, rnorm):
        rnorms.append(rnorm)
    ksp.setMonitor(monitor)

    ksp.setFromOptions()
    ksp.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)
    x = A.createVecRight()
    ksp.solve(b, x)

    rnorms = np.array(rnorms, dtype=float)
    if rnorms.size > 0 and rnorms[0] != 0.0:
        rrel = rnorms / rnorms[0]
    else:
        rrel = rnorms
    return rrel, ksp.getIterationNumber(), ksp

def assembled_pattern_nnz_from_sparsity_order(A, sparsity_order, cache=None):
    """Estimate assembled inverse nnz from fixed sparsity order using pattern of A^s."""
    if sparsity_order < 1:
        raise ValueError('sparsity_order must be >= 1')

    if cache is not None and sparsity_order in cache:
        return cache[sparsity_order]

    if sparsity_order == 1:
        nnz = int(A.getInfo()['nz_used'])
        if cache is not None:
            cache[sparsity_order] = nnz
        return nnz

    A_pow = A.copy()
    for _ in range(2, sparsity_order + 1):
        A_next = A_pow.matMult(A)
        A_pow.destroy()
        A_pow = A_next

    nnz = int(A_pow.getInfo()['nz_used'])
    A_pow.destroy()

    if cache is not None:
        cache[sparsity_order] = nnz
    return nnz

def richardson_matvec_equiv_per_iter(nnz_A, nnz_P=None, poly_order=None, matrix_free=False):
    if matrix_free:
        if poly_order is None:
            raise ValueError('poly_order required for matrix-free accounting')
        return 1.0 + float(poly_order)
    if nnz_P is None:
        raise ValueError('nnz_P required for assembled accounting')
    return 1.0 + float(nnz_P) / float(nnz_A)

In [ ]:
def plot_histories_vs_work(curves, title, xlabel='Matvec-equivalent work'):
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, work_x, rel_res in curves:
        if len(rel_res) == 0:
            continue
        ax.semilogy(work_x, rel_res, marker='o', ms=3, label=label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Relative residual')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Assembled 6th order, 1st order fixed sparsity GMRES polynomial vs GMRES(6)

For this section, we compare two different preconditioners inside an outer undamped Richardson iteration:
- Assembled, fixed sparsity GMRES polynomial in `PCPFLAREINV` (order 6, sparsity 1)
- GMRES(6)

Both are 6th order GMRES polynomials, but they differ substantially. `PCPFLAREINV` by default applies an assembled matrix approximation of the 6th order GMRES polynomial, restricted to first order sparsity. 

For example, with a 3rd-order monomial, the polynomial form is
$$
p_3(A)=c_0 I + c_1 A + c_2 A^2 + c_3 A^3.
$$

With assembled fixed-sparsity construction (`-pc_pflareinv_sparsity_order=1`), higher powers are truncated back to the sparsity pattern of $A$, e.g.
$$
\tilde p_3(A)=c_0 I + c_1 A + c_2\Pi_{\mathcal S_1}(A^2) + c_3\Pi_{\mathcal S_1}(A^3).
$$

So this section answers: with the same outer Richardson iteration counter, how does assembled stationary GMRES polynomial compare to a nonstationary GMRES(6)?

In [ ]:
# Outer undamped Richardson + assembled PCPFLAREINV
r_rich_pflare, it_rich_pflare, _ = run_solver_with_history(
    A, b, ksp_type='richardson', pc_type='pflareinv', max_it=200, rtol=1e-10
 )

# Outer undamped Richardson + fixed GMRES(6) apply
r_rich_gmres6, it_rich_gmres6, _ = run_richardson_with_fixed_gmres_apply(
    A, b, gmres_steps=6, max_it=200, rtol=1e-10
 )

iters_pflare = np.arange(1, len(r_rich_pflare) + 1, dtype=float)
iters_gmres6 = np.arange(1, len(r_rich_gmres6) + 1, dtype=float)

print(f'Richardson+PCPFLAREINV iterations: {it_rich_pflare}')
print(f'Richardson+GMRES(6)-apply iterations: {it_rich_gmres6}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(iters_pflare, r_rich_pflare, marker='o', ms=3, label='Richardson + PCPFLAREINV (assembled)')
ax.semilogy(iters_gmres6, r_rich_gmres6, marker='o', ms=3, label='Richardson + GMRES(6) apply')
ax.set_xlabel('Richardson iteration')
ax.set_ylabel('Relative residual')
ax.set_title('Assembled (6,1) vs GMRES(6) - by convergence')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Keep a plain GMRES baseline for later Section 6 work-normalized comparisons
r_gmres, it_gmres, _ = run_solver_with_history(
    A, b, ksp_type='gmres', pc_type='none', max_it=200, rtol=1e-10
 )
work_gmres = np.arange(1, len(r_gmres) + 1, dtype=float)

We can see that the assembled polynomial with first order fixed sparsity converges poorer than just using GMRES(6). This is to be expected given there are two sources of error in this approximation:
1. Applying fixed sparsity with the pattern of $A$ is very restrictive, considering the polynomials should have the sparsity of $A^6$.
2. GMRES(6) is minimising the residual at each iteration, where the stationary polynomial is built on a random rhs

But we need to examine the amount of work required to converge. If we measure the equivalent number of matrix-vector products with $A$ we see a different picture.

In [ ]:
# Work-based comparison for Point 3: assembled (6,1) vs GMRES(6)-apply
nnz_P_default = assembled_pattern_nnz_from_sparsity_order(A, 1)
mv_per_it_assembled = richardson_matvec_equiv_per_iter(
    nnz_A, nnz_P=nnz_P_default, matrix_free=False
)
mv_per_it_gmres6_apply = 1.0 + 6.0

work_pflare = mv_per_it_assembled * np.arange(1, len(r_rich_pflare) + 1, dtype=float)
work_gmres6_apply = mv_per_it_gmres6_apply * np.arange(1, len(r_rich_gmres6) + 1, dtype=float)

print(f'assembled (order=6, sparsity=1): matvec-equiv/iter={mv_per_it_assembled:.3f}')
print(f'GMRES(6)-apply + Richardson: matvec-equiv/iter={mv_per_it_gmres6_apply:.1f}')

plot_histories_vs_work(
    [
        ('Richardson + PCPFLAREINV (assembled 6,1)', work_pflare, r_rich_pflare),
        ('Richardson + GMRES(6) apply', work_gmres6_apply, r_rich_gmres6),
    ],
    title='Assembled (6,1) vs GMRES(6)- by matvec-equivalent work'
 )

We can see that when we measure the work required, the assembled fixed sparsity approximation beats the application of GMRES(6). This was why we introduced the fixed-sparsity approximation.

## 4. Increasing sparsity order

With polynomial order fixed at 6 (assembled), increasing `-pc_pflareinv_sparsity_order` enlarges the allowable nonzero pattern in the assembled approximate inverse.

Mechanistically:
- `-pc_pflareinv_sparsity_order=1` keeps a very local pattern
- Higher sparsity order allows longer-range couplings (entries associated with higher graph distance in powers of $A$)
- This usually gives a more faithful approximation to the full polynomial action, so Richardson convergence improves

Rather than look at the iteration count, we now plot the equivalent number of matrix-vector products required of $A$, as the sparsity order increases the nnzs in the assembled approximation increase. 


In [ ]:
sparsity_orders = [1, 2, 3, 4, 5, 6]
curves_sparse = []
nnz_sparse = []
nnz_cache = {}

for s in sparsity_orders:
    opts_s = {
        'pc_pflareinv_sparsity_order': s,
    }

    r_s, it_s, _ = run_solver_with_history(
        A, b, ksp_type='richardson', pc_type='pflareinv',
        pflare_opts=opts_s, max_it=200, rtol=1e-10
    )

    nnz_P_s = assembled_pattern_nnz_from_sparsity_order(A, s, cache=nnz_cache)
    mv_per_it_s = richardson_matvec_equiv_per_iter(nnz_A, nnz_P=nnz_P_s, matrix_free=False)
    work_s = mv_per_it_s * np.arange(1, len(r_s) + 1, dtype=float)

    curves_sparse.append((f'sparsity_order={s}', work_s, r_s))
    nnz_sparse.append(nnz_P_s)
    print(f'sparsity_order={s}: iters={it_s}, nnz(P)={nnz_P_s}, matvec-equiv/iter={mv_per_it_s:.3f}')

plot_histories_vs_work(
    curves_sparse,
    title='Assembled PCPFLAREINV, order 6: effect of sparsity_order'
 )

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sparsity_orders, nnz_sparse, '-o')
ax.set_xlabel('sparsity_order')
ax.set_ylabel('nnz of assembled polynomial inverse P')
ax.set_title('Assembled inverse fill increases with sparsity_order')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Firstly we can see the iteration count is decreasing as we increase `-pc_pflareinv_sparsity_order`, the approximation is getting closer to the true polynomial. Secondly we can see that the nnzs in the assembled approximation grows considerably with sparsity_order.

This leads to the fact that using a higher order sparsity in this case doesn't result in a cheaper method; the figure shows that we actually do more work with higher order sparsity to reach a given tolerance.  

## 5. Increase polynomial order with fixed sparsity=1

It may seem strange to use a higher polynomial order than fixed sparsity order, but this is key to the assembled approximation. In this test we keep `-pc_pflareinv_sparsity_order=1` fixed and increase polynomial order from 1 to 6.

This isolates the effect of polynomial degree under the same storage pattern. In other words, we enrich the polynomial model without letting the assembled matrix add broader fill.

Interpretation:
- Higher order gives more approximation power to match inverse-like behaviour
- Fixed sparsity prevents memory blow-up
- So we can still see order-driven improvement even under tight storage constraints

In [ ]:
orders = [1, 2, 3, 4, 5, 6, 7, 8]
curves_order_fixed_sparse = []

for m in orders:
    opts_m = {
        'pc_pflareinv_poly_order': m,
    }

    r_m, it_m, _ = run_solver_with_history(
        A, b, ksp_type='richardson', pc_type='pflareinv',
        pflare_opts=opts_m, max_it=200, rtol=1e-10
    )

    nnz_P_m = assembled_pattern_nnz_from_sparsity_order(A, 1)
    mv_per_it_m = richardson_matvec_equiv_per_iter(nnz_A, nnz_P=nnz_P_m, matrix_free=False)
    work_m = mv_per_it_m * np.arange(1, len(r_m) + 1, dtype=float)

    curves_order_fixed_sparse.append((f'order={m}, sparsity=1', work_m, r_m))
    print(f'order={m}: iters={it_m}, nnz(P)={nnz_P_m}, matvec-equiv/iter={mv_per_it_m:.3f}')

plot_histories_vs_work(
    curves_order_fixed_sparse,
    title='Assembled PCPFLAREINV: fixed sparsity=1, increasing order'
 )

We can see that the iteration count is plateauing; a 6th order polynomial with 1st order sparsity is a nice sweet spot. All the assembled approximations here cost the same to apply, given they share the same sparsity pattern. Assembling higher order, even with the same fixed sparsity order is more expensive, however.

## 6. Matrix-free polynomial application

A major advantage of polynomial approximate inverses is that they can be applied matrix-free. We don't have to store an assembled approximation to our stationary polynomial. The reason we default to a stored assembled approximation is that we explicitly need an assembled matrix when we form approximate ideal restrictors in AIRG. We will discuss this in a future notebook.

For now we can turn on the matrix-free application and do a direct comparison vs GMRES and the previous GMRES(6) applied with an undamped Richardson, isolating the effect of stationary approximate polynomial vs non-stationary. 

In [ ]:
mf_orders = [1, 2, 4, 6]

# Recompute GMRES baseline locally for this exact comparison
r_gmres_local, it_gmres_local, _ = run_solver_with_history(
    A, b, ksp_type='gmres', pc_type='none', max_it=300, rtol=1e-10
)
work_gmres_local = np.arange(1, len(r_gmres_local) + 1, dtype=float)
curves_mf = [('GMRES', work_gmres_local, r_gmres_local)]
print(f'GMRES: iters={it_gmres_local}, matvec-equiv/iter=1.0')

r_asm6, it_asm6, _ = run_solver_with_history(
    A, b, ksp_type='richardson', pc_type='pflareinv', max_it=200, rtol=1e-10
)
nnz_P_asm6 = assembled_pattern_nnz_from_sparsity_order(A, 1)
mv_per_it_asm6 = richardson_matvec_equiv_per_iter(nnz_A, nnz_P=nnz_P_asm6, matrix_free=False)
work_asm6 = mv_per_it_asm6 * np.arange(1, len(r_asm6) + 1, dtype=float)
curves_mf.append(('assembled order=6, sparsity=1', work_asm6, r_asm6))
print(f'assembled order=6, sparsity=1: iters={it_asm6}, matvec-equiv/iter={mv_per_it_asm6:.3f}')

# Add GMRES(6) apply with outer Richardson on same work axis
r_gmres6_apply, it_gmres6_apply, _ = run_richardson_with_fixed_gmres_apply(
    A, b, gmres_steps=6, max_it=200, rtol=1e-10
)
mv_per_it_gmres6_apply = 1.0 + 6.0
work_gmres6_apply = mv_per_it_gmres6_apply * np.arange(1, len(r_gmres6_apply) + 1, dtype=float)
curves_mf.append(('GMRES(6) apply + outer Richardson', work_gmres6_apply, r_gmres6_apply))
print(f'GMRES(6) apply + Richardson: iters={it_gmres6_apply}, matvec-equiv/iter={mv_per_it_gmres6_apply:.1f}')

for m in mf_orders:
    opts_mf = {
        'pc_pflareinv_poly_order': m,
        'pc_pflareinv_matrix_free': 1,
    }

    r_mf, it_mf, _ = run_solver_with_history(
        A, b, ksp_type='richardson', pc_type='pflareinv',
        pflare_opts=opts_mf, max_it=200, rtol=1e-10
    )

    mv_per_it_mf = richardson_matvec_equiv_per_iter(nnz_A, matrix_free=True, poly_order=m)
    work_mf = mv_per_it_mf * np.arange(1, len(r_mf) + 1, dtype=float)
    curves_mf.append((f'matrix-free order={m}', work_mf, r_mf))
    print(f'matrix-free order={m}: iters={it_mf}, matvec-equiv/iter={mv_per_it_mf:.1f}')

plot_histories_vs_work(
    curves_mf,
    title='Matrix-free PCPFLAREINV + assembled (6,1) + GMRES(6)-apply vs GMRES (2D upwind)'
)

We can see several effects here. Firstly as we increase the order of the polynomial applied matrix-free, the iteration count decreases. For this example however, the work calculation shows that repeatedly applying the 1st order matrix-free polynomial gives the best result from the matrix-free polynomials. 

We can do a direct comparison between GMRES(6) and the application of the 6th order stationary polynomial matrix-free. They are both 6th order GMRES polynomials, applied with an undamped Richardson. The only difference is GMRES(6) is minimising the residual at each iteration, whereas the stationary polynomial is not. 

We can see that GMRES(6) has a lower iteration count, and requires less work to solve. However we must remember that the stationary polynomial can be applied with matrix-vector products only, no dot products are required! This becomes important in future notebooks where we discuss parallelism. 

## Summary

- Stationary GMRES polynomials can form good approximate inverses. 
- If memory is not a concern, fixed sparsity assembled approximations can be built that require very little work to reach a given tolerance.
- If memory is limited, the stationary polynomials can be applied matrix-free, at the cost of more work.
- None of the stationary GMRES polynomials require dot products to apply. 

## 7. What's next?

We now move towards building a multigrid hierarchy.

**[03_cf_splitting.ipynb](03_cf_splitting.ipynb)**: Visualise the C/F splitting and understand the PMISR-DDC algorithm